# Longest Common Token Subsequence: Update 2026-04-15

This algorithm is developed and designed to compare two text sequences, especially of person, company, or region names or addresses: e.g. 'Alan Turing' vs. 'Alan Mathison Turing', 'United States of America' vs. 'U.S.A.', etc.

The code is based on the Longest Common Subsequence algorithm, where the order of characters/tokens/words matters and therefore a lower matching score is potentially calculated for some cases: e.g. 'Albert Einstein' vs. 'Einstein, Albert'. To overcome this challenge, the code locally performs the Longest Common Subsequence algorithm on every possible pair of tokens, whose time complexity still remains O(mn), where m = len(text1) and n = len(text2). 

And then, an algorithm for finding the maximum sum of values from non-overlapping rectangles is used to find the best matching result. First, I assume that a token pair represents a rectangle; that a rectangle represents a node in a graph; and that overlapping rectangles either on x-axis or y-axis are all directly connected adjacent nodes. This leads to a group or groups of graph, where each node has a value equivalent to the number of matched characters in a token pair. Now the problem is to find the best subset of nodes, using a brute force approach, that have the maximum sum of the node values, under the restriction that no two adjacent node values are used. This is a well-known house robbery problem but on a complex graph, rather than on a 1-d line or a tree. Unlike a house robbery problem on a 1-d line or a tree, which we can make the most out of dynamic programming and achieve O(n) time complexity, graphs that contain circles seems to break a kind of simplicity and prevent the use of dynamic programming, possibly leading to O(2^n) time complexity. To efficiently explore the nodes in the graph, I made the use of the depth-first-search approach. 

Overall, this algorithm is not ideal for long text comparison, because of potential O(2^n) time complexity due to the brute-force approach used. However, it is fast and useful for small text comparison, like entity names or addresses.

Please see the test result below on some examples. 

Bomsoo Kim

## Utils

In [1]:
import threading
import time

class TimeLimitExceededException(Exception):
    def __init__(self, error_message=''):
        super().__init__(error_message)

def use_threading_stop_event(func):# decorator
    def inner(*args, **kwargs):
        if ('time_limit_sec' in kwargs) and kwargs['time_limit_sec'] is not None:
            time_limit_sec = kwargs['time_limit_sec']
            stop_event = threading.Event()

            kwargs['threading_stop_event'] = stop_event # 'threading_stop_event' keyword argument must be defined in 'func'
            kwargs['threading_outputs'] = [None] # 'threading_outputs' keyword argument must be defined in 'func'

            my_thread = threading.Thread(target=func, args=args, kwargs=kwargs)

            my_thread.start()
            
            check_time_interval_sec = 0.000001
            tic = time.time()
            while True:
                if not my_thread.is_alive(): # if thread is completed, we don't need to wait
                    break
                elif (time.time() - tic) > time_limit_sec: # if time limit is exceeded, schedule a stop event
                    stop_event.set() # Signal stop

                time.sleep(check_time_interval_sec)

                if check_time_interval_sec < 1: # 1 sec
                    check_time_interval_sec += check_time_interval_sec # exponentially grow
                    check_time_interval_sec = min(1, check_time_interval_sec) # 1 sec

            my_thread.join()

            return kwargs['threading_outputs'][0]
        else:
            return func(*args, **kwargs)

    return inner


In [2]:
def flatten_trie_node(node, arr, matches):
    if not node:
        matches.append(list(arr))
        return
    
    for k in node.keys():
        arr.append(k)
        flatten_trie_node(node[k], arr, matches)
        arr.pop()
    return

## Levenshtein Distance
- [72. Edit Distance](https://leetcode.com/problems/edit-distance/)

In [3]:
def edit_distance(word1, word2): # https://leetcode.com/problems/edit-distance/
    """
    :type word1: str
    :type word2: str
    :rtype: int
    """
    dp = [[0]*(len(word2)+1) for _ in range(len(word1)+1)]
    
    for j in range(len(word2)-1, -1, -1):
        dp[-1][j] += 1 + dp[-1][j+1] # initial value
    
    for i in range(len(word1)-1, -1, -1):
        dp[i][-1] += 1 + dp[i+1][-1] # initial value
        
        for j in range(len(word2)-1, -1, -1):
            if word1[i] == word2[j]:
                dp[i][j] = dp[i+1][j+1]
            else:
                dp[i][j] = 1 + min(dp[i+1][j], dp[i][j+1], dp[i+1][j+1])
    return dp[0][0]

# if __name__=='__main__':
#     word1 = "horse"; word2 = "ros"
#     print(edit_distance(word1, word2))
#     # Output: 3
#     # Explanation: 
#     # horse -> rorse (replace 'h' with 'r')
#     # rorse -> rose (remove 'r')
#     # rose -> ros (remove 'e')
#     # Example 2:

#     word1 = "intention"; word2 = "execution"
#     print(edit_distance(word1, word2))
#     # Output: 5
#     # Explanation: 
#     # intention -> inention (remove 't')
#     # inention -> enention (replace 'i' with 'e')
#     # enention -> exention (replace 'n' with 'x')
#     # exention -> exection (replace 'n' with 'c')
#     # exection -> execution (insert 'u')   


## Longest common subsequence

In [4]:
def longest_common_subsequence(text1, text2): # https://leetcode.com/problems/longest-common-subsequence/
    if True:
        """
        :type text1: str
        :type text2: str
        :rtype: int
        """
        ### Update 03/02/2025: added matched sequence detection part
        ### Update 07/26/2022: all solutions are reviewed
        ### ref) 1143. Longest Common Subsequence: https://leetcode.com/problems/longest-common-subsequence/
        
        ### dynamic programming (bottom-up) with space optimizatoin ##################
        # if len(text1) < len(text2): # to make sure text2 is always shorter than text1
        #     text1, text2 = text2, text1

        # prev = [0]*(len(text2)+1) # space complexity: O(min(len(text1), len(text2)))
        # curr = [0]*(len(text2)+1) # space complexity: O(min(len(text1), len(text2)))
        # for i in range(len(text1)-1, -1, -1): # time complexity: O(len(text1) * len(text2))
        #     for j in range(len(text2)-1, -1, -1):
        #         if text1[i] == text2[j]:
        #             curr[j] = 1 + prev[j+1]
        #         else:
        #             curr[j] = max(prev[j], curr[j+1])
        #     prev, curr = curr, prev

        # return prev[0]
                
        ### dynamic programming (bottom-up) with matched sequence detection (03/02/2025) ###################
        dp = [[0]*(len(text2)+1) for _ in range(len(text1)+1)] # space complexity: O(len(text1) * len(text2))

        for i in range(len(text1)-1, -1, -1): # time complexity: O(len(text1) * len(text2))
            for j in range(len(text2)-1, -1, -1):
                if text1[i] == text2[j]:
                    dp[i][j] = 1 + dp[i+1][j+1]
                else:
                    dp[i][j] = max(dp[i+1][j], dp[i][j+1])

        #--- detect one of all possible matched sequences (03/02/2025) ---
        matches = []
        target = dp[0][0]
        j_start = 0
        for i in range(len(text1)):
            for j in range(j_start, len(text2)):
                if (text1[i] == text2[j]) and (target == dp[i][j]):
                    matches.append([i, j])
                    j_start = j + 1 # no need to see indexes less than matched j, because of the sequence order
                    target -= 1 # update the next target value
                    break # no need to see indexes great than matched j, because the matched is found!
            if target == 0:
                break
        assert dp[0][0] == len(matches), 'LCS value does not match the number of detected characters...'

        # return dp[0][0]
        return dp[0][0], matches

        ### dynamic programming (bottom-up) ###################
        # dp = [[0]*(len(text2)+1) for _ in range(len(text1)+1)] # space complexity: O(len(text1) * len(text2))

        # for i in range(len(text1)-1, -1, -1): # time complexity: O(len(text1) * len(text2))
        #     for j in range(len(text2)-1, -1, -1):
        #         if text1[i] == text2[j]:
        #             dp[i][j] = 1 + dp[i+1][j+1]
        #         else:
        #             dp[i][j] = max(dp[i+1][j], dp[i][j+1])

        # return dp[0][0]
                
        ### dynamix programming (top-down) ####################
    #     self.text1, self.text2 = text1, text2
    #     self.dp = {}
        
    #     return self.find_max_sub(0,0)
        
    # def find_max_sub(self, i, j):
    #     if (i,j) in self.dp:
    #         return self.dp[(i,j)]
        
    #     if i >= len(self.text1) or j >= len(self.text2):
    #         return 0
        
    #     if self.text1[i]==self.text2[j]:
    #         self.dp[(i,j)] = 1 + self.find_max_sub(i+1, j+1)
    #     else:
    #         self.dp[(i,j)] = max(self.find_max_sub(i, j+1), self.find_max_sub(i+1, j))
        
    #     return self.dp[(i,j)]

        ### Brad: solution with Longest Increasing Subsequence (Time Limt Exceeded) #########
        # arr = []
        # for i in range(len(text1)):
        #     for j in range(len(text2)-1, -1, -1):
        #         if text1[i] == text2[j]:
        #             arr.append(j)
                    
        # con = []
        # for j in arr: # Longest Increasing Subsequence
        #     if len(con) == 0 or con[-1] < j:
        #         con.append(j)
        #     else:
        #         for k in range(len(con)):
        #             if j <= con[k]:
        #                 con[k] = j
        #                 break
                        
        # return len(con)

# if __name__=='__main__':
#     text1 = "abcde"; text2 = "ace"
#     print(longest_common_subsequence(text1, text2)) # The longest common subsequence is "ace" and its length is 3

#     text1 = "abc"; text2 = "abc"
#     print(longest_common_subsequence(text1, text2)) # The longest common subsequence is "abc" and its length is 3

#     text1 = "abc"; text2 = "def"
#     print(longest_common_subsequence(text1, text2)) # There is no such common subsequence, so the result is 0

## Maximum value of non-overlapping intervals, with all possible solutions
- [1235. Maximum Profit in Job Scheduling](https://leetcode.com/problems/maximum-profit-in-job-scheduling/description/)
- [2008. Maximum Earnings From Taxi](https://leetcode.com/problems/maximum-earnings-from-taxi/description/)
- [1751. Maximum Number of Events That Can Be Attended II](https://leetcode.com/problems/maximum-number-of-events-that-can-be-attended-ii/description/)

In [5]:
# def get_all_non_overlapping_intervals(i, intervals, START=0, END=1, VALUE=2, dp=None, get_trie=True): # tested 2025-04-25
#     if i in dp:
#         return dp[i]

#     if i >= len(intervals):
#         return 0, {}
    
#     #--- binary search --------------------------
#     x, y = i, len(intervals)
#     while x + 1 < y:
#         m = (x + y) // 2
#         if intervals[i][END] < intervals[m][START]: # <, when [1,2] and [3,4] is considered non-overlapping
#         # if intervals[i][END] <= intervals[m][START]: # <=, when [1,2] and [2,3] is considered non-overlapping
#             y = m
#         else:
#             x = m

#     out_ = get_all_non_overlapping_intervals(x + 1, intervals, START=START, END=END, VALUE=VALUE, dp=dp, get_trie=get_trie)
#     out1 = intervals[i][VALUE] + out_[0], {i: out_[1]}
#     out2 = get_all_non_overlapping_intervals(i + 1, intervals, START=START, END=END, VALUE=VALUE, dp=dp, get_trie=get_trie)

#     max_val = max(out1[0], out2[0])
#     trie = {}
#     if get_trie:
#         if out1[0] == max_val:
#             trie.update(out1[1])
#         if out2[0] == max_val:
#             trie.update(out2[1])

#     dp[i] = max_val, trie
#     return dp[i]

# # if __name__=='__main__':
# #     # intervals = [[4,6,3],[2,5,7]]
# #     intervals = sorted([[1,3,3],[4,6,3],[2,5,6]])

# #     max_val, trie = get_all_non_overlapping_intervals(0, intervals, dp={})

# #     matches = []
# #     flatten_trie_node(trie, [], matches)
# #     print(matches)

## Union Find (Disjoint Sets)
- [1971. Find if Path Exists in Graph](https://leetcode.com/problems/find-if-path-exists-in-graph/)

In [6]:
class Union_Find:
    def __init__(self, num_points, edges):
        """
        
        """
        self.roots = [i for i in range(num_points)]
        self.ranks = [1 for _ in range(num_points)]

        for i, j in edges:
            self.union(i, j)
        return

    def find_root(self, i):
        if self.roots[i] != i:
            self.roots[i] =  self.find_root(self.roots[i])
        return self.roots[i]

    def union(self, i, j):
        x = self.find_root(i)
        y = self.find_root(j)
        if x != y:
            if self.ranks[x] > self.ranks[y]:
                self.roots[y] = x
            elif self.ranks[x] < self.ranks[y]:
                self.roots[x] = y
            else:
                self.roots[x] = y
                self.ranks[y] += self.ranks[x]
        return

# if __name__=='__main__':
#     values = [1,2,6,3,     1,3,4]
#     edges = [[0,1],[1,2],[0,3],[1,3],[2,3],     [4,5],[5,6]]
#     #---------------------------------------------------------
#     # values = [1,2,3]
#     # edges = []
#     #---------------------------------------------------------

#     num_points = len(values)

#     #--- find disjoint sets -------------------------
#     djs = Union_Find(num_points, edges) # disjoin sets

#     i_roots = set()
#     for i in range(num_points):
#         r = djs.find_root(i) # to make sure there is only one root for each group
#         i_roots.add(r)
#     print(f'i_roots = {i_roots}')


## House robbery
- [1192. Critical Connections in a Network](https://leetcode.com/problems/critical-connections-in-a-network)
- [337. House Robber III](https://leetcode.com/problems/house-robber-iii/)

In [7]:
# def find_critical_connections_in_network(prev, i, graph={}, ranks={}, critial_connections=[]): # UPDATE: 2026-03-16
#     ranks[i] = len(ranks)

#     min_rank = ranks[i]
#     if i in graph:
#         for j in graph[i]:
#             if j == prev:
#                 continue
#             elif j in ranks:
#                 rank_j = ranks[j]
#             else:
#                 rank_j = find_critical_connections_in_network(i, j, graph=graph, ranks=ranks, critial_connections=critial_connections)

#                 if ranks[i] < rank_j:
#                     critial_connections.append([i,j])

#             min_rank = min(min_rank, rank_j)

#     return min_rank

# if __name__=='__main__':
#     edges = []
#     #--- 1-d problem ----------------------
#     # edges = [[i,i+1] for i in range(8)]
#     #---- complete connection -------------
#     # edges = [[i,j] for i in range(15) for j in range(i+1, 15)]
#     #--- diamond and line -----------------------------------
#     # edges = [[0,1],[1,2],[0,3],[1,3],[2,3],     [4,5],[5,6]]
#     # #---diamond with line -----------------------------------
#     # edges = [[0,1],[1,2],[0,3],[1,3],[2,3],     [2,4],[4,5]]
#     #--------------------------------------

#     graph = {}
#     for i,j in edges:
#         graph.setdefault(i, []).append(j)
#         graph.setdefault(j, []).append(i)
#     # print(f'graph = {graph}')

#     critial_connections = []
#     _ = find_critical_connections_in_network(-1, 0, graph=graph, ranks={}, critial_connections=critial_connections)
#     print(f'critial_connections = {critial_connections}')

In [8]:
# def dfs(i, prev=-1, max_visit=-1, visits={}, graph={}):
#     if i not in visits:
#         visits[i] = len(visits)

#     max_visit = max(max_visit, visits[i])

#     for j in graph[i]:
#         if j == prev: # if j is the previouis 
#             continue
#         elif (j not in visits) or (visits[j] == max_visit + 1): # explore unvisited nodes or visit nodes in the same order
#             print(f'{i} --> {j} [OK]')
#             out = dfs(j, prev=i, max_visit=max_visit, visits=visits, graph=graph)
#             max_visit = max(max_visit, out['max_visit'])
#         elif visits[i] > visits[j]: # previously visits node: low rank contact
#             print(f'{i} --> {j} [Low rank contact]')
#             pass
#         elif visits[i] < visits[j]: # previously visits node: high rank contact
#             print(f'{i} --> {j} [High rank contact]')
#             pass
#         else:
#             raise(Exception('Brad error: impossible case, unless a self loop is allowed...'))

#     return {'max_visit':max_visit}


In [9]:
# def dfs_2(i, prev=-1, max_visit=-1, visits={}, graph={}, robbed=[], values=[], threading_stop_event=None):
#     if (threading_stop_event is not None) and threading_stop_event.is_set():
#         raise(TimeLimitExceededException(f'Brad error: time limit exceeded...')) 

#     if i not in visits:
#         visits[i] = len(visits)

#     # max_visit_0 = max_visit
#     max_visit_0 = max(max_visit, visits[i])

#     if any(robbed[j] > 0 for j in graph[i] if robbed[j] is not None): # check if there is any neighborhood robbed, among visited
#         modes = ['not rob'] # if any, then decide not to rob the current node
#     else:
#         modes = ['not rob', 'rob'] # if there is no neighborhood robbed, then there are two options: do not rob, or rob

#     max_val = 0
#     for mode in modes:
#         robbed[i] = 1 if mode == 'rob' else 0
#         # max_visit = max(max_visit_0, visits[i])
#         max_visit = max_visit_0 # initialize
#         sum_val = values[i] if mode == 'rob' else 0 # initialize

#         for j in graph[i]:
#             if j == prev: # if j is the previouis 
#                 continue
#             elif (j not in visits) or (visits[j] == max_visit + 1): # explore unvisited nodes or visit nodes in the same order
#                 # print(f'{i} --> {j} [OK]')
#                 out = dfs_2(j, prev=i, max_visit=max_visit, visits=visits, graph=graph, robbed=robbed, values=values, threading_stop_event=threading_stop_event)
#                 max_visit = max(max_visit, out['max_visit'])
#                 sum_val += out['max_val']
#             elif visits[i] > visits[j]: # previously visits node: low rank contact
#                 # print(f'{i} --> {j} [Low rank contact]')
#                 pass
#             elif visits[i] < visits[j]: # previously visits node: high rank contact
#                 # print(f'{i} --> {j} [High rank contact]')
#                 pass
#             else:
#                 raise(Exception('Brad error: impossible case, unless a self loop is allowed...'))

#         max_val = max(max_val, sum_val)
#     robbed[i] = None
#     return {'max_visit':max_visit, 'max_val':max_val}

# # if __name__=='__main__':
# #     values = [0,1,2,3,4,5,6]
# #     edges = [
# #         [0,1],[1,2],[2,3],[3,1],[3,0],[0,4],[4,5],[4,6],[0,6],
# #     ]

# #     values = [5, 2, 1, 1, 2, 12, 2, 2, 2, 1, 1, 1, 1, 2, 2, 7, 2, 2, 2, 1, 1, 1, 1, 1, 3]
# #     edges = [[0, 1], [0, 2], [0, 3], [0, 4], [0, 13], [0, 14], [1, 2], [1, 3], [1, 5], [1, 15], [1, 22], [2, 3], [2, 24], [3, 24], [4, 5], [4, 7], [4, 8], [4, 10], [4, 11], [4, 12], [4, 13], [4, 14], [5, 6], [5, 7], [5, 8], [5, 9], [5, 10], [5, 11], [5, 12], [5, 15], [5, 22], [5, 23], [6, 7], [6, 8], [6, 16], [6, 17], [6, 18], [7, 8], [7, 9], [7, 10], [7, 11], [7, 12], [7, 16], [7, 17], [7, 18], [8, 10], [8, 11], [8, 12], [8, 16], [8, 17], [8, 18], [9, 10], [9, 19], [9, 24], [10, 19], [10, 24], [11, 12], [11, 20], [11, 24], [12, 21], [12, 24], [13, 14], [13, 15], [13, 17], [13, 18], [13, 19], [13, 20], [13, 21], [14, 15], [14, 18], [14, 19], [14, 20], [14, 21], [15, 16], [15, 17], [15, 18], [15, 19], [15, 20], [15, 21], [15, 22], [15, 23], [16, 17], [16, 18], [17, 18], [18, 19], [18, 20], [18, 21], [19, 24], [20, 21], [20, 24], [21, 24], [22, 23], [22, 24], [23, 24]]

# #     graph = {}
# #     for i,j in edges:
# #         graph.setdefault(i, []).append(j)
# #         graph.setdefault(j, []).append(i)
# #     print(f'graph = {graph}')

# #     # visits={}
# #     # out = dfs(0, prev=-1, max_visit=-1, visits=visits, graph=graph)
# #     # print(out)
# #     # # dfs(1, prev=-1, max_visit=-1, visits=visits, graph=graph)

# #     print(f'values = {values}')
# #     robbed = [None]*len(values)
# #     print(f'robbed = {robbed}')
# #     for i in range(1):
# #         out2 = dfs_2(i, prev=-1, max_visit=-1, visits={}, graph=graph, robbed=robbed, values=values)
# #         print(out2)


In [10]:
def dfs_3(i, prev=-1, max_visit=-1, visits={}, graph={}, robbed=[], values=[], get_matches=False, threading_stop_event=None):
    if (threading_stop_event is not None) and threading_stop_event.is_set():
        raise(TimeLimitExceededException(f'Brad error: time limit exceeded...')) 

    if i not in visits:
        visits[i] = len(visits)

    max_visit_0 = max(max_visit, visits[i])

    if any(robbed[j] > 0 for j in graph[i] if robbed[j] is not None): # check if there is any neighborhood robbed, among visited
        modes = ['not rob'] # if any, then decide not to rob the current node
    else:
        modes = ['not rob', 'rob'] # if there is no neighborhood robbed, then there are two options: do not rob, or rob

    # max_val = 0
    max_val = float('-inf')
    idxs = []
    for mode in modes:
        robbed[i] = 1 if mode == 'rob' else 0
        max_visit = max_visit_0 # initialize
        sum_val = values[i] if mode == 'rob' else 0 # initialize
        if get_matches:
            idxs_0 = [[i]] if mode == 'rob' else [[]] # initialize

        for j in graph[i]:
            if j == prev: # if j is the previouis 
                continue
            elif (j not in visits) or (visits[j] == max_visit + 1): # explore unvisited nodes or visit nodes in the same order
                # print(f'{i} --> {j} [OK]')
                out = dfs_3(j, prev=i, max_visit=max_visit, visits=visits, graph=graph, robbed=robbed, values=values, get_matches=get_matches, threading_stop_event=threading_stop_event)
                max_visit = max(max_visit, out['max_visit'])
                sum_val += out['max_val']
                if get_matches:
                    idxs_0 = [ii1 + ii2 for ii1 in idxs_0 for ii2 in out['idxs']]

            elif visits[i] > visits[j]: # previously visits node: low rank contact
                # print(f'{i} --> {j} [Low rank contact]')
                pass
            elif visits[i] < visits[j]: # previously visits node: high rank contact
                # print(f'{i} --> {j} [High rank contact]')
                pass
            else:
                raise(Exception('Brad error: impossible case, unless a self loop is allowed...'))

        # max_val = max(max_val, sum_val)
        if max_val < sum_val:
            max_val = sum_val # replace
            if get_matches:
                idxs = idxs_0 # replace
        elif max_val == sum_val:
            if get_matches:
                idxs.extend(idxs_0) # add

    robbed[i] = None # reset
    return {'max_visit':max_visit, 'max_val':max_val, 'idxs':idxs}

# if __name__=='__main__':
#     values = [1,1,1,1]
#     edges = [[0,1],[1,2],[2,3]]

#     # values = [0,1,2,3,4,5,6]
#     # edges = [
#     #     [0,1],[1,2],[2,3],[3,1],[3,0],[0,4],[4,5],[4,6],[0,6],
#     # ]

#     # values = [5, 2, 1, 1, 2, 12, 2, 2, 2, 1, 1, 1, 1, 2, 2, 7, 2, 2, 2, 1, 1, 1, 1, 1, 3]
#     # edges = [[0, 1], [0, 2], [0, 3], [0, 4], [0, 13], [0, 14], [1, 2], [1, 3], [1, 5], [1, 15], [1, 22], [2, 3], [2, 24], [3, 24], [4, 5], [4, 7], [4, 8], [4, 10], [4, 11], [4, 12], [4, 13], [4, 14], [5, 6], [5, 7], [5, 8], [5, 9], [5, 10], [5, 11], [5, 12], [5, 15], [5, 22], [5, 23], [6, 7], [6, 8], [6, 16], [6, 17], [6, 18], [7, 8], [7, 9], [7, 10], [7, 11], [7, 12], [7, 16], [7, 17], [7, 18], [8, 10], [8, 11], [8, 12], [8, 16], [8, 17], [8, 18], [9, 10], [9, 19], [9, 24], [10, 19], [10, 24], [11, 12], [11, 20], [11, 24], [12, 21], [12, 24], [13, 14], [13, 15], [13, 17], [13, 18], [13, 19], [13, 20], [13, 21], [14, 15], [14, 18], [14, 19], [14, 20], [14, 21], [15, 16], [15, 17], [15, 18], [15, 19], [15, 20], [15, 21], [15, 22], [15, 23], [16, 17], [16, 18], [17, 18], [18, 19], [18, 20], [18, 21], [19, 24], [20, 21], [20, 24], [21, 24], [22, 23], [22, 24], [23, 24]]

#     graph = {}
#     for i,j in edges:
#         graph.setdefault(i, []).append(j)
#         graph.setdefault(j, []).append(i)
#     print(f'graph = {graph}')

#     # visits={}
#     # out = dfs(0, prev=-1, max_visit=-1, visits=visits, graph=graph)
#     # print(out)
#     # # dfs(1, prev=-1, max_visit=-1, visits=visits, graph=graph)

#     print(f'values = {values}')
#     robbed = [None]*len(values)
#     print(f'robbed = {robbed}')
#     for i in range(1):
#         # out2 = dfs_3(i, prev=-1, max_visit=-1, visits={}, graph=graph, robbed=robbed, values=values, get_matches=False)
#         out2 = dfs_3(i, prev=-1, max_visit=-1, visits={}, graph=graph, robbed=robbed, values=values, get_matches=True)
#         print(out2)


In [11]:
def get_max_house_robbery(values, edges, get_matches=False, debug=False, threading_stop_event=None):
    num_points = len(values)

    graph = {i:[] for i in range(len(values))}
    for i,j in edges:
        graph.setdefault(i, []).append(j)
        graph.setdefault(j, []).append(i)

    if debug:
        print(f'graph = {graph}')

    #--- find disjoint sets -------------------------
    djs = Union_Find(num_points, edges) # disjoin sets

    i_roots = set()
    for i in range(num_points):
        r = djs.find_root(i) # to make sure there is only one root for each group
        i_roots.add(r)

    if debug:
        print(f'i_roots = {i_roots}')

    #--------------------------------------
    robbed = [None]*len(values)
    if debug:
        print(f'robbed = {robbed}')

    # max_vals = []
    # for i in i_roots:
    #     out2 = dfs_2(i, prev=-1, max_visit=-1, visits={}, graph=graph, robbed=robbed, values=values, threading_stop_event=threading_stop_event)
    #     max_vals.append(out2['max_val'])

    # return sum(max_vals)

    max_vals = []
    idxs = [[]]
    for i in i_roots:
        out2 = dfs_3(i, prev=-1, max_visit=-1, visits={}, graph=graph, robbed=robbed, values=values, get_matches=get_matches, threading_stop_event=threading_stop_event)
        max_vals.append(out2['max_val'])
        if get_matches:
            idxs = [ii1 + ii2 for ii1 in idxs for ii2 in out2['idxs']]
 
    if not get_matches:
        idxs = None # reset
 
    return {'max_val':sum(max_vals), 'matches':idxs}

if __name__=='__main__':
    values = [1,2,3]
    edges = []
    #--- 1-d problem ----------------------
    # values = [1,2,3,1,9] # answer: 13
    # edges = [[i,i+1] for i in range(len(values)-1)]
    #---- complete connection -------------
    # values = [i for i in range(15)] # answer: 14 = (15 - 1)
    # edges = [[i,j] for i in range(len(values)) for j in range(i+1, len(values))]
    #--- only diamond -----------------------------------
    # values = [1,2,6,3] # answer: 7 (= 1 + 6)
    # edges = [[0,1],[1,2],[0,3],[1,3],[2,3]]
    #--- diamond and line -----------------------------------
    # values = [1,2,6,3,     1,3,4] # 12
    # edges = [[0,1],[1,2],[0,3],[1,3],[2,3],     [4,5],[5,6]]
    # #---diamond with line -----------------------------------
    # values = [1,2,6,3, 1,2] # 9
    # edges = [[0,1],[1,2],[0,3],[1,3],[2,3],     [2,4],[4,5]]
    #--- two diamonds with line -----------------------------------
    # values = [1,2,6,3,   3,1,2,   1,2] # 10
    # edges = [[0,1],[1,2],[0,3],[1,3],[2,3],   [2,4],[2,6],[4,6],[4,5],[5,6],     [2,7],[7,8]]
    #--- diamond and line -----------------------------------
    # values = [1,2,6,3,     1,3,4,     1,2,6,3,     ]
    # edges = [[0,1],[1,2],[0,3],[1,3],[2,3],     [4,5],[5,6],    [7,8],[8,9],[7,10],[8,10],[9,10], ]
    #--------------------------------------

    # out = get_max_house_robbery(values, edges, get_matches=False, debug=False)
    out = get_max_house_robbery(values, edges, get_matches=True, debug=False)
    print(f'out = {out}')

out = {'max_val': 6, 'matches': [[0, 1, 2]]}


## Longest common subsequence, with all possible solutions
- [1143. Longest Common Subsequence](https://leetcode.com/problems/longest-common-subsequence/description/)

In [12]:
def lcs(i, j, text1, text2, dp=None, get_trie=True): # tested 2025-04-25
    if (i,j) in dp:
        return dp[(i,j)]
   
    if i >= len(text1) or j >= len(text2):
        return 0, {}
    
    if text1[i] == text2[j]:       
        out_ = lcs(i+1, j+1, text1, text2, dp=dp, get_trie=get_trie)
        out0 = 1 + out_[0], {(i,j):out_[1]}
    else:
        out0 = -1, {}
    out1 = lcs(i, j+1, text1, text2, dp=dp, get_trie=get_trie)
    out2 = lcs(i+1, j, text1, text2, dp=dp, get_trie=get_trie)
    
    max_val = max(out0[0], out1[0], out2[0])
    trie = {}
    if get_trie:
        if out0[0] == max_val:
            trie.update(out0[1])
        if out1[0] == max_val:
            trie.update(out1[1])
        if out2[0] == max_val:
            trie.update(out2[1])
    
    dp[(i,j)] = max_val, trie
    return dp[(i,j)]


## Longest common token subsequence (Greedy & Brute force)

- helping functions

In [13]:
def is_word_char(c, word_char_set=set('0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')):
    return c in word_char_set

# if __name__=='__main__':
#     print(is_word_char('c'))
#     print(is_word_char('-'))
#     print(is_word_char('#'))

In [14]:
def find_tokens(text, word_char_set=set('0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')): # UPDATE: 2026-03-16
    #--- forward cumulutive sum of characters -----------------------
    csum = [0] * len(text) # initialize to zeros
    for i in range(len(text)-1, -1, -1):
        if is_word_char(text[i], word_char_set=word_char_set):
            if i == len(text)-1: # if the very last charcter
                csum[i] = 1
            else:
                csum[i] = csum[i+1] + 1

    #--- token connectivity = root ----------------------------------
    root = [None] * len(text)
    iroot, cnt = {}, 0
    for i in range(len(text)):
        if is_word_char(text[i], word_char_set=word_char_set):
            if i == 0 or not is_word_char(text[i-1], word_char_set=word_char_set): # if the initial character of each token
                i_initial = i
                iroot[i], cnt = cnt, cnt + 1
            root[i] = i_initial

    return csum, root, iroot

# if __name__=="__main__":
#     print(find_tokens('hello world!'))
#     # ([5, 4, 3, 2, 1, 0, 5, 4, 3, 2, 1, 0], [0, 0, 0, 0, 0, None, 6, 6, 6, 6, 6, None], {0: 0, 6: 1})

- main

In [15]:
# def longest_common_token_subsequence_OLD(text1, text2, word_char_set=set('0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')): # UPDATE: 2026-03-16
#     all_matches = []

#     csum1, root1, iroot1 = find_tokens(text1, word_char_set=word_char_set)
#     csum2, root2, iroot2 = find_tokens(text2, word_char_set=word_char_set)

#     #--- token-by-token longest common subsequence: greedy for each token-token pair --------------------------------------
#     for i in iroot1.keys():
#         for j in iroot2.keys():
#             _text1_ = text1[i:(i + csum1[i])]
#             _text2_ = text2[j:(j + csum2[j])]

#             val, trie = lcs(0, 0, _text1_, _text2_, dp={})

#             matches = []
#             flatten_trie_node(trie, [], matches)
#             # print(f'_text1_ = {_text1_}, _text2_ = {_text2_}, matches = {matches}')

#             for pairs in matches:
#                 if pairs:
#                     all_matches.append([(i+x, j+y) for x,y in pairs])

#     #--- sort along x-axis and find non-overlapping rectanlges over x-axis ------------------------------------------------
#     #--- there can be still some recangles overlapping over y-axis, but they will be removed in the next step -------------
#     AXIS = 0 # x
#     SCORE = 0
#     intervals_0 = [[pairs[0][AXIS], pairs[-1][AXIS], SCORE, k] for k, pairs in enumerate(all_matches)] # SCORE = 0, to find all possible group of non-overlapping rectangles
#     intervals_0 = sorted(intervals_0)

#     _, trie_0 = get_all_non_overlapping_intervals(0, intervals_0, dp={})
#     matches_0 = []
#     flatten_trie_node(trie_0, [], matches_0)
#     # print(matches_0)

#     #--- sort along y-axis and find non-overlapping rectangles over y-axis ------------------------------------------------
#     ans = []
#     seen = set()
#     for mm in matches_0:
#         mm_orig_0 = [intervals_0[i][3] for i in mm]
#         # print(mm, mm_orig_0)

#         AXIS = 1 # y
#         intervals_1 = [[all_matches[k][0][AXIS], all_matches[k][-1][AXIS], len(all_matches[k]), k] for k in mm_orig_0] # SCORE = len(all_matches[k])
#         intervals_1 = sorted(intervals_1)

#         _, trie_1 = get_all_non_overlapping_intervals(0, intervals_1, dp={})
#         matches_1 = []
#         flatten_trie_node(trie_1, [], matches_1)

#         for mm_1 in matches_1:
#             mm_orig_1 = [intervals_1[i][3] for i in mm_1]

#             key_seen = tuple(sorted(mm_orig_1))
#             if key_seen not in seen:
#                 score = sum(len(all_matches[i]) for i in mm_orig_1)
                
#                 while ans and ans[-1][0] < score:
#                     _ = ans.pop()
                
#                 if (not ans) or (ans[-1][0] == score):
#                     ans.append((score, mm_orig_1))

#                 seen.add(key_seen)

#     #--- find matched results ------------------------------------------------------------------
#     outs = []
#     for _, option in ans:
#         match1, match2 = [-1]*len(text1), [-1]*len(text2) # initialize
#         for k, i in enumerate(option):
#             pairs = all_matches[i]
#             for x, y in pairs:
#                 if match1[x] < 0:
#                     match1[x] = k
#                 else:
#                     raise(Exception('Brad error: a critical issue occurred. There is an overlap...'))
                
#                 if match2[y] < 0:
#                     match2[y] = k
#                 else:
#                     raise(Exception('Brad error: a critical issue occurred. There is an overlap...'))

#         outs.append({'match1':match1, 'match2':match2,})

#     out = {'csum1':csum1, 'root1':root1, 'iroot1':iroot1, 'csum2':csum2, 'root2':root2, 'iroot2':iroot2, 'matches': outs}

#     return out

# # if __name__=='__main__':
#     # # text1 = 'xab'
#     # # text2 = 'axb'
#     # # text1 = 'xxoxxx'
#     # # text2 = 'xxx'
#     # # text1 = 'xxxxx'
#     # # text2 = 'oxxx'
#     # # text1 = 'oxxxxx'
#     # # text2 = 'xxx'
#     # # text1 = 'xxccxab'
#     # # text2 = 'axb'
#     # # text1 = 'xxxxxxyxxx'
#     # # text2 = 'xyyxx'
#     # text1 = 'abcabc'
#     # text2 = 'abc abc'

#     # val, trie = lcs(0, 0, text1, text2, dp={})

#     # matches = []
#     # flatten_trie_node(trie, [], matches)
#     # # print(matches)

In [16]:
@use_threading_stop_event
def longest_common_token_subsequence( # UPDATE: 2026-04-15
    text1, text2,
    word_char_set=set('0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'),
    get_matches=False,
    debug=False, 
    time_limit_sec=None, threading_stop_event=None, threading_outputs=[None],
    ): # UPDATE: 2026-03-16
    csum1, root1, iroot1 = find_tokens(text1, word_char_set=word_char_set)
    csum2, root2, iroot2 = find_tokens(text2, word_char_set=word_char_set)

    #--- token-by-token longest common subsequence: greedy for each token-token pair --------------------------------------
    all_matches = []
    all_matches_full = {}
    seen = set()
    for i in iroot1.keys():
        for j in iroot2.keys():
            _text1_ = text1[i:(i + csum1[i])]
            _text2_ = text2[j:(j + csum2[j])]

            val, trie = lcs(0, 0, _text1_, _text2_, dp={}) # partially greedy approach

            matches = []
            flatten_trie_node(trie, [], matches)
            # print(f'_text1_ = {_text1_}, _text2_ = {_text2_}, matches = {matches}')

            for pairs in matches:
                # if pairs:
                    # all_matches.append([(i+x, j+y) for x,y in pairs])
                if pairs and (pairs[0][0] == 0 or pairs[0][1] == 0): # exclude this match: e.g. 'xxxAAyyy' vs 'zzAAwww', but this is okay: 'AAyyy' vs 'zzAAwww'
                    (x0, y0), (x1, y1) = pairs[0], pairs[-1] # only top left & bottom right points (rectangle)
                    xyxy_val_ij = (i+x0, j+y0, i+x1, j+y1, val, i, j)
                    if xyxy_val_ij not in seen:
                        all_matches.append(xyxy_val_ij)
                        seen.add(xyxy_val_ij)

                    if get_matches:
                        all_matches_full.setdefault(xyxy_val_ij, []).append([(i+x, j+y) for x,y in pairs])

    if debug:
        print(f'len(all_matches) = {len(all_matches)}')
        # print(f'all_matches = {all_matches}')

    #-----------------
    values = []
    edges = []
    for i in range(len(all_matches)):
        x0, y0, x1, y1, val, ri, rj = all_matches[i]
        values.append(val)

        for j in range(i+1, len(all_matches)):
            if i == j:
                continue

            a0, b0, a1, b1, _, si, sj = all_matches[j]

            if (x0 <= a1 and a0 <= x1) or (y0 <= b1 and b0 <= y1): # if there is any overlap on either x-axis OR y-axis
                edges.append([i,j])
            elif (ri == si) and (rj == sj): # if there is no ovelap between two recangles, but they are in the same token, then they can not co-exist
                edges.append([i,j])

    if debug:
        print(f'len(values) = {len(values)}')
        # print(f'values = {values}')
        print(f'len(edges) = {len(edges)}')
        # print(f'edges = {edges}')

    #-----------------
    out = get_max_house_robbery(values, edges, get_matches=get_matches, debug=False, threading_stop_event=threading_stop_event)
    if debug:
        print(f'out = {out}')

    # --- find matched results ------------------------------------------------------------------
    if get_matches:
        outs = []
        for one_matches in out['matches']:
            option_all = [[]]
            for i in one_matches:
                option_all = [ii1 + [(i,j)] for ii1 in option_all for j in range(len(all_matches_full[all_matches[i]]))] # find all possible combinations
 
            for option in option_all:
                match1, match2 = [-1]*len(text1), [-1]*len(text2) # initialize
                # for k, i in enumerate(option):
                for k, (i,j) in enumerate(option):
                    # pairs = all_matches[i]
                    pairs = all_matches_full[all_matches[i]][j]
                    for x, y in pairs:
                        if match1[x] < 0:
                            match1[x] = k
                        else:
                            raise(Exception('Brad error: a critical issue occurred. There is an overlap...'))
                      
                        if match2[y] < 0:
                            match2[y] = k
                        else:
                            raise(Exception('Brad error: a critical issue occurred. There is an overlap...'))
 
                outs.append({'match1':match1, 'match2':match2,})
    else:
        outs = None

    output = {'csum1':csum1, 'root1':root1, 'iroot1':iroot1, 'csum2':csum2, 'root2':root2, 'iroot2':iroot2, 'num_matches': out['max_val'], 'matches':outs} # UPDATE: 2026-04-15

    threading_outputs[0] = output
    return output

# if __name__=='__main__':
#     text1 = 'abcabc'
#     text2 = 'abc abc'
#     # text1 = 'aaaaaaaaaaaaaaaaaaa'
#     # text2 = 'aaa aaa aaaaaaaaa aaa aa'
#     # text1, text2 = ('Smith Bro., 123 Main Street, Unit 5, Scanton, A1B 2C3', 'Smith Brothers, 132 Main St. Unit 5, Scanton, A1B 2C4')
#     # text1 = '1122'
#     # text2 = '22 11'
#     text1 = '1122'
#     text2 = '2211'

#     # out = longest_common_token_subsequence(text1.lower(), text2.lower(), debug=True)
#     out = longest_common_token_subsequence(text1.lower(), text2.lower(), time_limit_sec=5, debug=True)
#     print(out)
#     if out is not None:
#         print(f"num_matches = {out['num_matches']}")

- miscellaneous

In [17]:
def token_matched_all_or_partial_but_in_full(root1, match1):
    rcnt, mcnt = {}, {}
    for r, m in zip(root1, match1):
        if r is not None:
            if r not in rcnt:
                rcnt[r] = 0
            if r not in mcnt:
                mcnt[r] = 0
                
            rcnt[r] += 1
            if m >= 0:
                mcnt[r] += 1

    perfect_matched = all(mcnt[k] == v for k,v in rcnt.items())
    partial_token_matched = all(mcnt[k]==0 or mcnt[k] == v for k,v in rcnt.items())
    return perfect_matched, partial_token_matched

In [18]:
def count_consecutive_initials(ii, i_not_set):
    conn = [0] * len(ii)
    for k,v in ii.items():
        if k not in i_not_set:
            conn[v] = 1

    cnt = 0
    for i in range(len(conn)):
        if conn[i] > 0 and (i == 0 or conn[i-1] == 0):
            cnt += 1

    return cnt

In [19]:
def is_two_texts_same(csum1, iroot1, match1, csum2, iroot2, match2):
    ii_remaining_1 = set(iroot1.keys()).intersection([i for i, (t,m) in enumerate(zip(csum1, match1)) if t > 0 and m < 0]) # if token and not assigned
    ii_remaining_2 = set(iroot2.keys()).intersection([i for i, (t,m) in enumerate(zip(csum2, match2)) if t > 0 and m < 0]) # if token and not assigned            

    cnt1 = count_consecutive_initials(iroot1, ii_remaining_1)
    cnt2 = count_consecutive_initials(iroot2, ii_remaining_2)

    is_same = (
        (len(ii_remaining_1) == 0 and (len(iroot1) <= len(iroot2) - len(ii_remaining_2))) or
        (len(ii_remaining_2) == 0 and (len(iroot2) <= len(iroot1) - len(ii_remaining_1)))
        ) and (cnt1 == 1 and cnt2 == 1)
    return is_same


In [20]:
def get_impurity_score(arr, type=['Gini','Entropy'][1]): # https://datasciencedojo.com/blog/gini-index-and-entropy/
    import math

    Ps = {}
    for c in arr:
        if c not in Ps:
            Ps[c] = 0
        Ps[c] += 1.0 / len(arr)

    if type == 'Gini':
        score = 1
        for c,p in Ps.items():
            score -= p**2

    elif type == 'Entropy':
        score = 0
        for c,p in Ps.items():

            score += -p*math.log2(p)
    else:
        raise(ValueError)

    return score

# if __name__=='__main__':
#     print(get_impurity_score([1,1,1,1,1,1], type=['Gini','Entropy'][0])) # 0
#     print(get_impurity_score([1,1,1,3,3,3], type=['Gini','Entropy'][0])) # 0.5
#     print(get_impurity_score([1,1,2,2,3,3], type=['Gini','Entropy'][0])) # 0.6666666666666665
#     print(get_impurity_score([1,1,1,1,1,1], type=['Gini','Entropy'][1])) # 0
#     print(get_impurity_score([1,1,1,3,3,3], type=['Gini','Entropy'][1])) # 1.0
#     print(get_impurity_score([1,1,2,2,3,3], type=['Gini','Entropy'][1])) # 1.584962500721156

In [21]:
def get_average_impurity_score(root, match, type=['Gini','Entropy'][1]):
    arrs = {}
    for r, m in zip(root, match):
        if (r is not None) and (m >= 0):
            arrs.setdefault(r,[]).append(m)

    scores, cnts = [], []
    for arr in arrs.values():
        scores.append(get_impurity_score(arr, type=type))
        cnts.append(len(arr))

    if cnts:
        score = 1.0 * sum(c*s for c,s in zip(cnts,scores)) / sum(cnts)
    else:
        score = 0

    return score

In [22]:
def log_matched_results(text1, text2, match1, match2):
    match_out1 = f"{text1}\n{''.join(['^' if n >=0 else ' ' for n in match1])}\n{''.join([chr(ord('A')+n) if n >=0 else ' ' for n in match1])}"
    match_out2 = f"{text2}\n{''.join(['^' if n >=0 else ' ' for n in match2])}\n{''.join([chr(ord('A')+n) if n >=0 else ' ' for n in match2])}"
    log = f'{match_out1}\n{match_out2}'
    return log

# TEST

In [23]:
if __name__=='__main__':
    import time

    for text1, text2 in [
        # ('abc', 'abcabc'),
        # ('abc abc', 'abcabc'),
        # ('abc', 'aabbcc'),
        # ('abc abc', 'aabbcc'),
        # ('abc', 'aaaaaabc'),
        # ('xab', 'axb'),
        # ('axbc', 'abcx'),
        # ('abc abc', 'abcabcxa'),
        # ('aecxef ghi', 'aec xef ghi'),
        ('aaabbb ccc', 'aaa bbbccc'),

        ('Alan Turing', 'Alan Mathison Turing'),
        ('Albert Einstein', 'Einstein, Albert'),
        ('Tom Edison', 'Thomas Edison'),
        ('Bomsoo Kim', 'Bom Soo Kim'),
        ('Bomsoo Kim', 'Kim, Bom-soo'),
        ('Bomsoo Kim', 'Bomsoo P. Kim'),
        ('Bomsoo Peter Kim', 'Bomsoo P. Kim'),
        ('Brad Pitt', 'Bomsoo Kim'),
        ('Tingting Wong', 'Ting Ting Wong'),
        ('Yun Ying Jin', 'Yunying Jin'),
        ('International I. Company Ltd', 'Int. Inn Co. Limited'),
        ('International Inn Company Ltd', 'I. I. Co. Limited'),
        ('Green Construction Corporation Ltd.', 'Green Construction Co., Limited'),
        ('United States of America', 'U.S.A.'),
        ('United States of America', 'USA'),
        ('New York, NY ', 'NY, NY'),
        ('1122 Hunter St.','11 22 Hunter St.'),
        ('1122 Hunter St.','22 11 Hunter St.'),
        ('1122 Hunter St.','2211 Hunter St.'),
        ('Smith Bro., 123 Main Street, Unit 5, Scanton, A1B 2C3', 'Smith Brothers, 132 Main St. Unit 5, Scanton, A1B 2C4'),
        # ('Empire State Building, 20 W 34th St., New York, NY 10001 United States of America (Tel. 123-456-7890)', '20 WEST 34TH STREET, EMPIRE STATE BLD. 44TH FLOOR ROOM#2, NY, NY 10001-00000 U.S.A.'), # !!!!!!!!!!!!!!!! NY needs to be captured
        ]:

        print(f'##############################################################################################################################')
        tic = time.time()
        # out = longest_common_token_subsequence(text1.lower(), text2.lower(), get_matches=False, time_limit_sec=15, debug=False)
        out = longest_common_token_subsequence(text1.lower(), text2.lower(), get_matches=True, time_limit_sec=15, debug=False)
        toc = time.time()

        csum1, root1, iroot1 = out['csum1'], out['root1'], out['iroot1']
        csum2, root2, iroot2 = out['csum2'], out['root2'], out['iroot2']

        for i, row in enumerate(out['matches']):
            match1, match2 = row['match1'], row['match2']

            log = log_matched_results(text1, text2, match1, match2)

            score1 = sum(n>=0 for n in match1) / sum(r is not None for r in root1)
            score2 = sum(n>=0 for n in match2) / sum(r is not None for r in root2)

            print(f"---- [{i+1}/{len(out['matches'])}] ---------------------------------------------------")
            print(log)
            print(f"score1 = {score1}, score2 = {score2}, avg(scores) = {0.5*(score1 + score2)}")
            print(f"num_matches_1 = {sum(n>=0 for n in match1)}")
            print(f"num_matches_2 = {sum(n>=0 for n in match2)}")

        print(f'\nTIME ELAPSED = {toc - tic} sec')

        # print(f'============================================================================================')
        # tic = time.time()
        # out_OLD = longest_common_token_subsequence_OLD(text1.lower(), text2.lower())
        # toc = time.time()

        # csum1, root1, iroot1 = out_OLD['csum1'], out_OLD['root1'], out_OLD['iroot1']
        # csum2, root2, iroot2 = out_OLD['csum2'], out_OLD['root2'], out_OLD['iroot2']

        # for i, row in enumerate(out_OLD['matches']):
        #     match1, match2 = row['match1'], row['match2']

        #     log = log_matched_results(text1, text2, match1, match2)

        #     score1 = sum(n>=0 for n in match1) / sum(r is not None for r in root1)
        #     score2 = sum(n>=0 for n in match2) / sum(r is not None for r in root2)

        #     print(f"---- [{i+1}/{len(out_OLD['matches'])}] ---------------------------------------------------")
        #     print(log)
        #     print(f"score1 = {score1}, score2 = {score2}, avg(scores) = {0.5*(score1 + score2)}")
        #     print(f"num_matches_1 = {sum(n>=0 for n in match1)}")
        #     print(f"num_matches_2 = {sum(n>=0 for n in match2)}")

        # print(f'\nTIME ELAPSED = {toc - tic} sec')

        # print(f'#####################################################################################')
        # out = longest_common_token_subsequence_OLD(text1.lower(), text2.lower())

        # csum1, root1, iroot1 = out['csum1'], out['root1'], out['iroot1']
        # csum2, root2, iroot2 = out['csum2'], out['root2'], out['iroot2']

        # for i, row in enumerate(out['matches']):
        #     match1, match2 = row['match1'], row['match2']
            
        #     log = log_matched_results(text1, text2, match1, match2)            

        #     #------------------------------------------------------------------------------------------------
        #     is_same = is_two_texts_same(csum1, iroot1, match1, csum2, iroot2, match2)
            
        #     perfect_matched1, partial_token_matched1 = token_matched_all_or_partial_but_in_full(root1, match1)
        #     perfect_matched2, partial_token_matched2 = token_matched_all_or_partial_but_in_full(root2, match2)

        #     print(f"---- [{i+1}/{len(out['matches'])}] ---------------------------------------------------")
        #     print(log)
        #     print(f'RESULT: {is_same or (partial_token_matched1 and partial_token_matched2 and (perfect_matched1 or perfect_matched2))}')


##############################################################################################################################
---- [1/1] ---------------------------------------------------
aaabbb ccc
^^^^^^ ^^^
AAABBB CCC
aaa bbbccc
^^^ ^^^^^^
AAA BBBCCC
score1 = 1.0, score2 = 1.0, avg(scores) = 1.0
num_matches_1 = 9
num_matches_2 = 9

TIME ELAPSED = 0.0 sec
##############################################################################################################################
---- [1/1] ---------------------------------------------------
Alan Turing
^^^^ ^^^^^^
AAAA BBBBBB
Alan Mathison Turing
^^^^          ^^^^^^
AAAA          BBBBBB
score1 = 1.0, score2 = 0.5555555555555556, avg(scores) = 0.7777777777777778
num_matches_1 = 10
num_matches_2 = 10

TIME ELAPSED = 0.004106044769287109 sec
##############################################################################################################################
---- [1/1] ---------------------------------------------------
Albe

# END